## This is a set of convenience utilities that help you manipulate and clean up coco files

## Function: load_coco_to_dataframe_with_attributes

This function restructures a coco file to a dataframe, allowing an analyst to more conveniently inspect
and analyze its contents. The dataframe that is returned has one record for each annotation (object)
in the image.

WARNING: BOUNDING BOX AND X/Y COORDINATES: I'm not completely confident that the bbox to x/y height/width logic is correct. Need to test/verify

## Function: filter_coco_by_annotations

When manually annotating in tools like CVAT, you may be left with images where you don't annotate anything
for whatever reason. In some cases that might be legitimate and you want negative examples for training,
but in other cases maybe not - for example you might have skipped a lot of redundant images. This will 
strip those out.

Not implemented yet but was the original reason for this function: pass in a specific annotation and filter
for only images with that annotation - the reason for this is because if we want to add annotations to a set
of images that have already been annotated for another purpose, there may be ambiguity as to whether other
previously annotated images don't have the new annotation because they don't have that object, OR if they
just haven't been looked at.


## Function: copy_coco_images

Once you have a coco file with the contents you want, you may find that the directory that houses the images has
far more images than you need. This will create a dir (if necessary) and copy/paste the images in that coco 
file to keep things more organized.

## Function: copy_missing_images

Get a list of jpgs in a directory, and copy them to a target directory if those same images can't be found in a 3rd reference directory.
I can't remember why I needed to do this... 


## split_coco_annotations

Splits a coco file into test/train (not implemented: validate). This is probably redundant with the code in
create_coco_detector_partitions.




In [9]:
#https://pypi.org/project/pycocotools/
#https://gist.github.com/interactivetech/c2913317603b79c02ff49fa9824f1104
from pycocotools.coco import COCO
from pprint import pprint
import pandas as pd
import json
import sys
import os
import shutil




In [6]:
## define environment specific variables

# root directory
base_loc = '/mnt/d/projects_working_directories/202505_mapswipe_test2'

# directory where coco file(s) can be found
coco_dir = f'{base_loc}/data/coco/'

# input coco file
annFile = f'{coco_dir}/20250511_test_unfiltered.json'


# ouput coco file name
output_file = f'{coco_dir}/20250511_test_filtered.json'


# only necessary for copy_coco_images()
coco_json_path = '/mnt/d/projects_working_directories/202505_mapswipe_test2/package1_baseline/coco/{}'.format('20250511_test_filtered.json')
source_img_path = '/mnt/d/projects_working_directories/202505_mapswipe_test2/data/img'
dest_img_path = '/mnt/d/projects_working_directories/202505_mapswipe_test2/package1_baseline/img'




In [7]:
# initialize COCO api for instance annotations
coco=COCO(annFile)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [8]:
## look at one example from the input coco file
for key in coco.anns.keys():
    pprint(coco.anns[key])
    break

{'area': 148492.67699999997,
 'attributes': {'building_quality': 'high',
                'building_type': 'mixed',
                'has_soft_story': 'yes',
                'num_of_stories': 3.0,
                'occluded': False,
                'overhang_type': 'non-structurel',
                'rotation': 0.0},
 'bbox': [150.43, 87.05, 309.45, 479.86],
 'category_id': 1,
 'id': 20,
 'image_id': 31,
 'iscrowd': 0,
 'segmentation': []}


In [5]:
def load_coco_to_dataframe_with_attributes(coco_json_path):
    # Load COCO JSON
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    # Create image_id to file_name mapping
    images_df = pd.DataFrame(coco['images'])
    images_df = images_df.rename(columns={'id': 'image_id'})
    images_df = images_df[['image_id', 'file_name']]

    # Create category_id → category_name mapping
    categories_df = pd.DataFrame(coco['categories'])
    categories_df = categories_df.rename(columns={'id': 'category_id', 'name': 'category_name'})
    categories_df = categories_df[['category_id', 'category_name']]

    # Load annotations
    annotations_df = pd.DataFrame(coco['annotations'])
    annotations_df = annotations_df.rename(columns={'id': 'annotation_id'})

    # Get attributes
    if 'attributes' in annotations_df.columns:
        attr_df = annotations_df['attributes'].apply(pd.Series)
        annotations_df = pd.concat([annotations_df.drop(columns=['attributes']), attr_df], axis=1)

    # Merge image file names into annotations
    merged_df = annotations_df.merge(images_df, on='image_id', how='left')

    # Merge category names into annotations
    merged_df = merged_df.merge(categories_df, on='category_id', how='left')

    # Expand bbox list [x, y, width, height] into separate columns
    bbox_df = pd.DataFrame(merged_df['bbox'].tolist(), columns=['x', 'y', 'width', 'height'])
    merged_df = pd.concat([merged_df, bbox_df], axis=1)
    
    #merged_df = pd.concat([merged_df.drop(columns=['bbox']), bbox_df], axis=1)

    # Rearrange columns for readability
    static_cols = [
        'annotation_id', 'file_name', 'image_id', 'category_id', 'category_name',
        'x', 'y', 'width', 'height', 'iscrowd', 'area'
    ]
    # Add attribute columns (if any) at the end
    attribute_cols = [col for col in merged_df.columns if col not in static_cols and col not in ['segmentation']]
    final_cols = static_cols + attribute_cols

    merged_df = merged_df[final_cols]

    return merged_df

# Example usage
df = load_coco_to_dataframe_with_attributes(annFile)
df.head()

,annotation_id,file_name,image_id,category_id,category_name,x,y,width,height,iscrowd,area,bbox,building_quality,building_type,has_soft_story,num_of_stories,overhang_type,occluded,rotation
0,1,1455826318258897_right.jpg,2,1,building,149.61,415.65,305.20,216.85,0,66182.6200,"[149.61, 415.65, 305.2, 216.85]",high,commercial,no,1.0,non-structurel,False,0.0
1,2,1455826318258897_right.jpg,2,1,building,519.06,320.68,503.97,355.39,0,179105.8983,"[519.06, 320.68, 503.97, 355.39]",high,commercial,no,1.0,non-structurel,False,0.0
2,3,1458116667999603_left.jpg,3,1,building,159.65,435.73,524.05,471.77,0,247231.0685,"[159.65, 435.73, 524.05, 471.77]",high,residential,no,1.0,non-structurel,False,0.0
3,4,146199971451052_left.jpg,9,1,building,235.84,219.02,306.24,379.95,0,116355.8880,"[235.84, 219.02, 306.24, 379.95]",high,industrial,no,13.0,none,True,0.0
4,5,146199971451052_left.jpg,9,1,building,0.86,462.82,449.13,255.93,0,114945.8409,"[0.86, 462.82, 449.13, 255.93]",high,commercial,no,1.0,non-structurel,False,0.0


In [16]:


def filter_coco_by_annotations(input_path, output_path):
    ''' strip out items without images '''
    
    with open(input_path, 'r') as f:
        coco = json.load(f)

    # Get image IDs that have at least one annotation
    annotated_image_ids = {ann['image_id'] for ann in coco['annotations']}

    # Filter images that are in the set of annotated IDs
    filtered_images = [img for img in coco['images'] if img['id'] in annotated_image_ids]

    # Build new COCO structure
    filtered_coco = {
        'images': filtered_images,
        'annotations': filtered_annotations,
        'categories': coco['categories']  # categories stay the same
    }

    # Copy 'info' and 'licenses' if they exist
    if 'info' in coco:
        filtered_coco['info'] = coco['info']
    if 'licenses' in coco:
        filtered_coco['licenses'] = coco['licenses']

    # Write output
    with open(output_path, 'w') as f:
        json.dump(filtered_coco, f, indent=2)


filter_coco_by_annotations(annFile, output_file)


In [22]:


def copy_coco_images(coco_json_path, source_dir, dest_dir):
    ''' given a list of images in a coco file, copy and paste'''
    
    # Load COCO file
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    # Get unique image file names
    image_files = {img['file_name'] for img in coco['images']}

    print(f"Found {len(image_files)} unique images in COCO file.")

    # Ensure destination directory exists
    os.makedirs(dest_dir, exist_ok=True)

    copied = 0
    missing = []

    for filename in image_files:
        src_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(dest_dir, filename)

        # Ensure subdirectories exist (if COCO file_name includes paths)
        #os.makedirs(os.path.dirname(dest_path), exist_ok=True)

        if os.path.isfile(src_path):
            shutil.copy2(src_path, dest_path)
            copied += 1
        else:
            missing.append(filename)

    print(f"Copied {copied} images to {dest_dir}")

    if missing:
        print(f"Warning: {len(missing)} images not found in source_dir:")
        for m in missing:
            print(f"  - {m}")



copy_coco_images(coco_json_path, source_img_path, dest_img_path)


Found 14 unique images in COCO file.
Copied 14 images to /mnt/d/projects_working_directories/202505_mapswipe_test2/package1_baseline/img


In [24]:
def copy_missing_images(source_dir, reference_dir, target_dir):
    '''get a list of jpgs in a directory, and copy them to a target directory if 
    those same images can't be found in a 3rd reference directory'''
    
    # Get list of all JPGs in source_dir
    source_images = {f for f in os.listdir(source_dir) if f.lower().endswith('.jpg')}

    # Get list of all JPGs in reference_dir
    reference_images = {f for f in os.listdir(reference_dir) if f.lower().endswith('.jpg')}

    # Images to copy = in source but NOT in reference
    images_to_copy = source_images - reference_images

    print(f"Found {len(source_images)} JPGs in source.")
    print(f"Found {len(reference_images)} JPGs in reference.")
    print(f"{len(images_to_copy)} images to copy.")

    # Ensure target directory exists
    os.makedirs(target_dir, exist_ok=True)

    copied = 0
    for filename in images_to_copy:
        src_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(target_dir, filename)

        shutil.copy2(src_path, dest_path)
        copied += 1
        if copied == 100:
            break

    print(f"Copied {copied} images to {target_dir}")

source_dir = '/mnt/d/projects_working_directories/202505_mapswipe_test2/data/img'
reference_dir = '/mnt/d/projects_working_directories/202505_mapswipe_test2/package1_baseline/img'
target_dir = '/mnt/d/projects_working_directories/202505_mapswipe_test2/package1_baseline/100_images_for_inferences'

copy_missing_images(source_dir, reference_dir, target_dir)

Found 714 JPGs in source.
Found 143 JPGs in reference.
571 images to copy.
Copied 100 images to /mnt/d/projects_working_directories/202505_mapswipe_test2/package1_baseline/100_images_for_inferences


## split a coco file into train and test

In [1]:
import json
import random
import os

def split_coco_annotations(input_json, output_dir, split_ratio=0.8, seed=42):
    with open(input_json, 'r') as f:
        coco = json.load(f)

    random.seed(seed)

    images = coco['images']
    annotations = coco['annotations']
    categories = coco['categories']

    # Shuffle images and split
    random.shuffle(images)
    split_index = int(len(images) * split_ratio)
    train_images = images[:split_index]
    test_images = images[split_index:]

    # Create lookup for image IDs
    train_ids = set(img['id'] for img in train_images)
    test_ids = set(img['id'] for img in test_images)

    # Split annotations based on image IDs
    train_annotations = [ann for ann in annotations if ann['image_id'] in train_ids]
    test_annotations = [ann for ann in annotations if ann['image_id'] in test_ids]

    # Create output dicts
    train_coco = {
        'images': train_images,
        'annotations': train_annotations,
        'categories': categories
    }

    test_coco = {
        'images': test_images,
        'annotations': test_annotations,
        'categories': categories
    }

    # Write files
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, 'train.json'), 'w') as f:
        json.dump(train_coco, f, indent=2)

    with open(os.path.join(output_dir, 'test.json'), 'w') as f:
        json.dump(test_coco, f, indent=2)

    print(f"Split completed: {len(train_images)} train images, {len(test_images)} test images.")




In [3]:
split_coco_annotations(
    input_json='/mnt/c/temp/sample.json',
    output_dir='/mnt/c/temp/',
    split_ratio=0.8  # 80% train, 20% test
)

Split completed: 80 train images, 20 test images.
